In [1]:
%load_ext autoreload
%autoreload 2

from functools import partial

import moe
from tests import utils as test_utils

import jax
import jax.numpy as jnp
from jax import random
from jax.sharding import PartitionSpec as P, NamedSharding
import tune_jax
import numpy as np

tune_jax.logger.setLevel("INFO")

In [2]:
idx = jax.random.randint(jax.random.key(0), (8192,), minval=0, maxval=4)

In [3]:
moe.utils.add_indices(jnp.array([1, 3, 5]), jnp.array([0, 7, 2]), 10)

Array([         3,          3,          3,          3,          3,
                3,          3,          5,          5, 2147483647,
       2147483647, 2147483647, 2147483647, 2147483647, 2147483647,
       2147483647, 2147483647, 2147483647, 2147483647, 2147483647,
       2147483647, 2147483647, 2147483647, 2147483647, 2147483647,
       2147483647, 2147483647, 2147483647, 2147483647, 2147483647],      dtype=int32)

In [4]:
jnp.bincount(moe.utils.add_indices(jnp.array([1, 3, 5]), jnp.array([0, 7, 2]), 10), length=16)

Array([0, 0, 0, 7, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [5]:
jnp.bincount(jnp.arange(12), length=2)

Array([1, 1], dtype=int32)

# actual moe testing

In [14]:
n_devices = jax.device_count()
mesh = jax.make_mesh((n_devices,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.sharding.set_mesh(mesh)

In [17]:
x, meta = test_utils.generate_data(4096, 7168, device_num=n_devices, axis_name="x")

In [18]:
g = 32
all_idxs = jax.random.randint(jax.random.key(0), (2 * x.shape[0],), minval=0, maxval=g)
x2 = moe.core.run_moe(x, all_idxs, axis_name="x", experts_num=g, multiple=8)

On TPU_0(process=0,(0,0,0,0)) at mesh coordinates (x,) = (0,):
[[         7          7          7          7          7         15         15         23         23         23         23         23         23         23         31         31 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647]
 [         7          7          7          7          7         15         15         23 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647]
 [         7         15         15         15         23         23         23         23         23         31         31         31         31         31         31         31 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647 2147483647]
 [  

In [19]:
x2.shape

(4096, 2, 8, 896)

In [20]:
x.shape

(4096, 8, 896)

In [21]:
jnp.sum(jnp.abs(jnp.sum(x2[:, :1, ...], axis=1) - x))

Array(0, dtype=bfloat16)

In [22]:
x.at[1024-15:1024, 0, ...].get(out_sharding=P())

Array([[972, 972, 972, ..., 972, 972, 972],
       [972, 972, 972, ..., 972, 972, 972],
       [976, 976, 976, ..., 976, 976, 976],
       ...,
       [1008, 1008, 1008, ..., 1008, 1008, 1008],
       [1008, 1008, 1008, ..., 1008, 1008, 1008],
       [1024, 1024, 1024, ..., 1024, 1024, 1024]], dtype=bfloat16)

In [23]:
x2.at[1024-15:1024, 0, 0, ...].get(out_sharding=P())

Array([[972, 972, 972, ..., 972, 972, 972],
       [972, 972, 972, ..., 972, 972, 972],
       [976, 976, 976, ..., 976, 976, 976],
       ...,
       [1008, 1008, 1008, ..., 1008, 1008, 1008],
       [1008, 1008, 1008, ..., 1008, 1008, 1008],
       [1024, 1024, 1024, ..., 1024, 1024, 1024]], dtype=bfloat16)

In [24]:
jnp.cumsum(partial(jnp.bincount, length=g)(all_idxs), axis=-1)

Array([ 253,  499,  741,  975, 1251, 1491, 1753, 2006, 2254, 2501, 2758,
       3030, 3301, 3553, 3804, 4073, 4328, 4567, 4854, 5105, 5350, 5608,
       5861, 6129, 6343, 6640, 6903, 7184, 7436, 7675, 7933, 8192],      dtype=int32)

In [25]:
def print_me(i):
  print(x2.at[i - 1:i + 2, 0, 0, ...].get(out_sharding=P()))
  print(x.at[i - 1:i + 2, 0, ...].get(out_sharding=P()))

In [26]:
print_me(868)

[[380 380 380 ... 380 380 380]
 [384 384 384 ... 384 384 384]
 [384 384 384 ... 384 384 384]]
[[380 380 380 ... 380 380 380]
 [384 384 384 ... 384 384 384]
 [384 384 384 ... 384 384 384]]


In [27]:
print_me(4086)

[[4064 4064 4064 ... 4064 4064 4064]
 [4064 4064 4064 ... 4064 4064 4064]
 [4064 4064 4064 ... 4064 4064 4064]]
[[4064 4064 4064 ... 4064 4064 4064]
 [4064 4064 4064 ... 4064 4064 4064]
 [4064 4064 4064 ... 4064 4064 4064]]


In [28]:
jnp.where(jnp.sum(jnp.abs(jnp.sum(x2[:, :1, ...], axis=1) - x), axis=-1).at[:, 0].get(out_sharding=P()))

(Array([], shape=(0,), dtype=int32),)

In [85]:
(1024 - 9) / 1024

0.9912109375

In [15]:
n_devices = jax.device_count()
axis_name = "x"
mesh = jax.make_mesh((n_devices,), (axis_name,), axis_types=(jax.sharding.AxisType.Explicit,))

experts_per_tok = 2

with jax.sharding.set_mesh(mesh):
  n, k = 16, 2048
  g = 32  # experts
  x, ra2a_meta = test_utils.generate_data(n, k, n_devices, axis_name="x")
  del ra2a_meta
  all_idxs = jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g)
  out = moe.core.run_moe(x, all_idxs, axis_name="x", experts_num=n_devices)
  x_new = np.array(out[:, 0, :, :])
  np.testing.assert_allclose(x, x_new)

local_group_sizes = On TPU_0(process=0,(0,0,0,0)) at mesh coordinates (x,) = (0,):
[0]

On TPU_1(process=0,(1,0,0,0)) at mesh coordinates (x,) = (1,):
[1]

On TPU_3(process=0,(1,1,0,0)) at mesh coordinates (x,) = (2,):
[0]

On TPU_2(process=0,(0,1,0,0)) at mesh coordinates (x,) = (3,):
[3]

y.shape = (16, 8, 256)
y.shape = (16, 8, 256)
x_sort.shape = (8, 8, 256)
x.shape = (4, 2, 8, 256)


In [19]:
all_idxs

Array([ 5, 16,  8, 29,  3, 19, 16, 31, 21, 27,  4, 10, 24,  3, 27,  3, 22,
       18, 10, 29, 27, 24, 28, 19,  5, 29, 20,  9, 24,  4,  1, 20],      dtype=int32)

In [16]:
x

Array([[[1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        ...,
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1]],

       [[2, 2, 2, ..., 2, 2, 2],
        [2, 2, 2, ..., 2, 2, 2],
        [2, 2, 2, ..., 2, 2, 2],
        ...,
        [2, 2, 2, ..., 2, 2, 2],
        [2, 2, 2, ..., 2, 2, 2],
        [2, 2, 2, ..., 2, 2, 2]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[13, 13, 13, ..., 13, 13, 13],
        [13, 13, 13, ..., 13, 13, 13],
        [13, 13, 13, ..., 13, 13, 13],
        ...,
        [13, 13, 13, ..., 13, 13, 13],
        [13, 13, 13, ..., 13, 13, 13],
        [13, 13, 13, ..., 13, 13, 13]],

       [[14, 14, 14, ..., 14, 14, 14],
        [14, 14, 14, ..., 14, 14, 14],
        [14, 14, 14, 

In [22]:
jnp.sum(jnp.abs(x_new - x))

Array(0, dtype=bfloat16)

In [ ]:
x.s

(16, 8, 1)

In [9]:
x

Array([[[1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1]],

       [[2],
        [2],
        [2],
        [2],
        [2],
        [2],
        [2],
        [2]],

       [[0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0]],

       [[3],
        [3],
        [3],
        [3],
        [3],
        [3],
        [3],
        [3]],

       [[6],
        [6],
        [6],
        [6],
        [6],
        [6],
        [6],
        [6]],

       [[4],
        [4],
        [4],
        [4],
        [4],
        [4],
        [4],
        [4]],

       [[5],
        [5],
        [5],
        [5],
        [5],
        [5],
        [5],
        [5]],

       [[7],
        [7],
        [7],
        [7],
        [7],
        [7],
        [7],
        [7]],

       [[10],
        [10],
        [10],
        [10],
        [10],
        [10],
        [10],
        [10]],

       [[11],
        [11],
        [1

In [10]:
x_new

array([[[1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1]],

       [[2],
        [2],
        [2],
        [2],
        [2],
        [2],
        [2],
        [2]],

       [[0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0]],

       [[3],
        [3],
        [3],
        [3],
        [3],
        [3],
        [3],
        [3]],

       [[6],
        [6],
        [6],
        [6],
        [6],
        [6],
        [6],
        [6]],

       [[4],
        [4],
        [4],
        [4],
        [4],
        [4],
        [4],
        [4]],

       [[5],
        [5],
        [5],
        [5],
        [5],
        [5],
        [5],
        [5]],

       [[7],
        [7],
        [7],
        [7],
        [7],
        [7],
        [7],
        [7]],

       [[0],
        [1],
        [2],
        [1],
        [3],
        [0],
        [2],
        [3]],

       [[0],
        [1],
        [2],
       

# mesh experiments

In [2]:
n = jax.device_count()
mesh = jax.make_mesh((n,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.sharding.set_mesh(mesh)

In [3]:
fn = lambda: moe.utils.empty((1024, 1024), jnp.bfloat16, P(None, "x"))

In [4]:
fn_ = tune_jax.tune(fn)
fn_()

Compiling...:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling tpu:   0%|          | 0/5 [00:00<?, ?it/s]

Saving optimization profile to `/tmp/tuning_profile_2025-11-17_19:20:23_91fy8g4e`


Profiling tpu: 100%|██████████| 5/5 [00:04<00:00,  1.16it/s]


Array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=bfloat16)

# ra2a 2d

In [2]:
n_devices = jax.device_count()
mesh = jax.make_mesh((n_devices,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.set_mesh(mesh)

In [20]:
x_sort, input_offsets, send_sizes, output_offsets, recv_sizes = test_utils.generate_data(8 * 8 * 4096, 4096, n_devices, multiple=8)
x_sort = x_sort.reshape((x_sort.shape[0], -1)).astype(jnp.bfloat16)

In [29]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None), check_vma=False)
def test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  #output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  output = moe.utils.empty((2 * x.shape[0],) + x.shape[1:], x.dtype)
  with jax.named_scope("start"):
    out = moe.ra2a.ra2a_2d(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x", multiple=8)
  with jax.named_scope("jax.lax.ragged_all_to_all"):
    out2 = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
  return out, out2

In [34]:
out, out2 = test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes)
err = jnp.sum(jnp.abs(out - out2))
print(f"{err = }")

err = Array(0, dtype=bfloat16)


In [35]:
with jax.profiler.trace("/tmp/ra2a"):
  for _ in range(3):
    out, out2 = jax.block_until_ready(test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes))

# compute on

In [ ]:
import jax
import jax.numpy as jnp
from jax.experimental.compute_on import compute_on

import numpy as np

@jax.jit
@compute_on("tpu_sparsecore")
def my_gather(x, idx):
  return x[idx, ...]

x = jnp.ones((8192, 4096))
idx = jnp.argsort(np.random.randn(x.shape[0]))

In [6]:
with jax.profiler.trace("/tmp/compute_on"):
  for _ in range(3):
    jax.block_until_ready(my_gather(x, idx))